In [5]:
# Option 1 - local
!pip install sentence-transformers pandas numpy scikit-learn faiss-cpu

# Option 2 - OpenAI
!pip install openai pandas numpy scikit-learn faiss-cpu


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize
import faiss  # optional: only if you want indexing
from typing import List, Tuple, Optional

# 1) Create embeddings for a corpus and return a DataFrame
def build_embeddings_df(
    texts: List[str],
    model_name: str = "all-MiniLM-L6-v2",  # fast, good baseline
    batch_size: int = 64,
    normalize_embeddings: bool = True
) -> pd.DataFrame:
    """
    Returns DataFrame with columns: index (int), text (str), embedding (list of float32)
    """
    model = SentenceTransformer(model_name)
    # encode returns ndarray shape (N, D)
    embeddings = model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)
    # cast to float32
    embeddings = embeddings.astype(np.float32)
    if normalize_embeddings:
        # L2-normalize embeddings so cosine similarity = dot product
        embeddings = normalize(embeddings, norm='l2', axis=1).astype(np.float32)

    # Build DataFrame
    df = pd.DataFrame({
        "index": np.arange(len(texts)),
        "text": texts,
        # convert embedding rows to lists for easy JSON/CSV saving if needed
        "embedding": [emb.tolist() for emb in embeddings]
    })
    # keep embeddings as array separately if you want to use FAISS / speed
    df.index = df["index"]
    return df, embeddings  # return both df and numpy matrix for indexing

# 2) Brute-force nearest neighbors using DataFrame (small-medium corpuses)

# 3) Faiss index for larger corpuses (fast approximate or exact depending on index)


In [7]:
texts = [
    "How to bake sourdough bread",
    "Sourdough starter care",
    "How to train a dog",
    "Dog training basics for puppies",
    "Best coffee beans 2025"
]
df, emb_matrix = build_embeddings_df(texts)
df

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,index,text,embedding
index,,,
0,0,How to bake sourdough bread,"[0.024785660207271576, 0.03715621307492256, -0..."
1,1,Sourdough starter care,"[-0.03089788928627968, -0.0294658150523901, 0...."
2,2,How to train a dog,"[0.0033735360484570265, -0.039680760353803635,..."
3,3,Dog training basics for puppies,"[-0.059012748301029205, -0.06209912896156311, ..."
4,4,Best coffee beans 2025,"[-0.07767204940319061, -0.023800255730748177, ..."


In [17]:
def query_top_k_bruteforce(
    query: str,
    df: pd.DataFrame,
    model: SentenceTransformer,
    top_k: int = 5,
    normalize_query: bool = True
) -> pd.DataFrame:
    """
    Returns top_k rows from df with similarity scores.
    Expects df['embedding'] to contain list[float].
    """
    q_emb = model.encode([query], convert_to_numpy=True).astype(np.float32)
    if normalize_query:
        q_emb = normalize(q_emb, norm='l2', axis=1).astype(np.float32)
    # build matrix of embeddings
    emb_matrix = np.vstack(df['embedding'].apply(np.array).values).astype(np.float32)  # shape (N,D)
    #print(df['embedding'])
    #print(emb_matrix )
    # cosine similarity = dot product when vectors are L2 normalized
    sims = (emb_matrix @ q_emb[0]).astype(float)  # shape (N,)
    df_result = df.copy()
    df_result["score"] = sims
    df_result = df_result.sort_values("score", ascending=False).head(top_k)
    return df_result[["index", "text", "score"]]


In [18]:
print(query_top_k_bruteforce("how to care for sourdough starter", df, model, top_k=3))

       index                         text     score
index                                              
1          1       Sourdough starter care  0.892588
0          0  How to bake sourdough bread  0.481942
4          4       Best coffee beans 2025  0.168129


In [20]:
def build_faiss_index(embeddings: np.ndarray, use_gpu: bool = False) -> faiss.Index:
    """
    embeddings: np.ndarray (N, D) dtype float32 and L2-normalized if cosine via inner product
    Returns a FAISS index configured for inner product search (cosine when normalized).
    """
    n, d = embeddings.shape
    # We'll use IndexFlatIP for exact inner-product search (fast, memory O(N))
    index = faiss.IndexFlatIP(d)   # inner product
    index.add(embeddings)          # add vectors
    return index




<faiss.swigfaiss_avx2.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x78d806344750> >


In [23]:
def query_faiss(
    query: str,
    model: SentenceTransformer,
    index: faiss.Index,
    df: pd.DataFrame,
    top_k: int = 5,
    normalize_query: bool = True
) -> pd.DataFrame:
    q_emb = model.encode([query], convert_to_numpy=True).astype(np.float32)
    if normalize_query:
        q_emb = normalize(q_emb, norm='l2', axis=1).astype(np.float32)
    D, I = index.search(q_emb, top_k)  # D: scores, I: indices
    results = []
    for score, idx in zip(D[0], I[0]):
        results.append({
            "index": int(df.loc[idx, "index"]),
            "text": df.loc[idx, "text"],
            "score": float(score)
        })
    return pd.DataFrame(results)

In [24]:
texts = [
    "How to bake sourdough bread",
    "Sourdough starter care",
    "How to train a dog",
    "Dog training basics for puppies",
    "Best coffee beans 2025"
]
df, emb_matrix = build_embeddings_df(texts)
# build FAISS index for fast retrieval
model = SentenceTransformer("all-MiniLM-L6-v2")
faiss_index = build_faiss_index(emb_matrix)
print(query_faiss("how to care for sourdough starter", model, faiss_index, df, top_k=3))
# or use brute force
print(query_top_k_bruteforce("how to care for sourdough starter", df, model, top_k=3))


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   index                         text     score
0      1       Sourdough starter care  0.892588
1      0  How to bake sourdough bread  0.481942
2      4       Best coffee beans 2025  0.168129
       index                         text     score
index                                              
1          1       Sourdough starter care  0.892588
0          0  How to bake sourdough bread  0.481942
4          4       Best coffee beans 2025  0.168129
